# Forma mentis networks from association and valence task

## Imports

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import spacy
from emoatlas import EmoScores
from scipy import stats
import nltk
nltk.data.path.append('/Users/emma/nltk_data')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from nltk import word_tokenize, pos_tag

In [ ]:
## ITALIAN ##
nlp = spacy.load("it_core_news_lg")

emo = EmoScores(language="italian", spacy_model='it_core_news_lg')

In [ ]:
## ENLGISH ##
nlp = spacy.load("en_core_web_lg")

emo = EmoScores(language="english", spacy_model='en_core_web_lg')

In [ ]:
pd.set_option('display.max_rows', None)

## Text cleaning

In [ ]:
def lemmatize_text_it(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

In [ ]:
# function to clean the text: remove punctuation, spaces, numbers and whitespaces. make it lowercase
def text_cleaning(df):
    # clean columns that have words inside
    columns_to_clean = ['cue_word', 'association_1', 'association_2', 'association_3']
    for col in columns_to_clean:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.lower()
                .str.replace(r'[\d\W_]+', ' ', regex=True)
                .str.strip()
                # .apply(lemmatize_text_it)
            )
    return df

## Dataframe preparation

In [ ]:
# function to transform the df from wide to long format for later analysis
def build_association_df(df):

    # preserve original cue word for later indexing (before being renamed)
    df['cue_word_original'] = df['cue_word']  

    # rename cue and valence to fit wide format
    df_long = df.rename(columns={
        'cue_word': 'association_0',
        'valence_cue_word': 'valence_association_0'
    })

    # transform the df from wide to long format based on their columns and unique identifiers ('participant_id', 'cue_word_original')
    df_long = pd.wide_to_long(df_long, stubnames=['association', 'valence_association'], 
                          i=['participant_id', 'cue_word_original'], j='index', sep='_', suffix=r'\d+').reset_index()


    # sort df for the cue_word and the index of the association
    df_long = df_long.sort_values(by=['cue_word_original', 'index']).reset_index(drop=True)
    
    # rename columns to be more explicit
    df_long = df_long.rename(columns={'cue_word_original' : 'cue_word', 
                                      'association': 'word', 
                                      'valence_association': 'valence'})
    
    # df_long = df_long[['participant_id', 'cue_word', 'word', 'valence']]
    df_long = df_long.dropna()

    df_long = df_long[(df_long['word'] != 'nan') & (df_long['word'] != '')]

    ## !!! OPTIONAL: FILTERED !!! filter out words that were given by only one participant for each cue word ###
    # df_long = df_long.groupby(['cue_word', 'word']).filter(lambda g: g['participant_id'].nunique() > 1)

    return df_long

## Valence categorization



### Plot of the distribution of the categorization values

In [ ]:
# function to plot valence categories distrubution
def plot_valence_distribution(df):
    plt.figure(figsize=(6,5))

    # count the number of occurrences of each valence category
    counts = df['valence_cat'].value_counts().sort_index()
    tick_positions = range(len(counts)) 
    counts.plot(kind='bar')

    plt.title('Distribuzione Categorie Valenza')
    plt.xlabel('Categoria')
    plt.ylabel('Conteggio')
    plt.xticks(tick_positions, labels=['-1 (Negativo)', '0 (Neutro)', '+1 (Positivo)'], rotation=0)

    plt.tight_layout()
    plt.show()

### Categorization for single participant

We assign categorical labels to numeric valence scores based on the range (max - min) of their distribution:

- If γ (gamma) > 1:
The participant has used the full scale meaningfully. Concepts at the extremes (minimum, maximum, or more extreme than Q1/Q3) are labeled as negative or positive. Others are labeled neutral. We use inclusive quartiles.

- If γ ≤ 1:
The range of used values is very narrow (e.g., only 3–4). We can't reliably distinguish between positive and negative valences, so we assign the neutral class to all.

γ = 1 and γ < 1 are treated the same way to avoid forcing distinctions in very narrow ranges.

In [ ]:
def categorize_valence_single(df):
    
    # convert to NumPy array
    arr = np.array(df['valence'])

    # compute value range (gamma)
    minv, maxv = arr.min(), arr.max()
    gamma = maxv - minv

    # if values are too close together, return all neutral --> if gamma ≤ 1 (gamma = 0 and gamma < 1 included)
    # if gamma <= 1:
    #     return pd.Series([0] * len(arr), index=valence_series.index)
    if gamma <= 1:
        df['valence_cat'] = 0

    # if gamma > 1: use inclusive quartiles to classify
    Q1, Q3 = np.percentile(arr, [25, 75])

    #  defines classification logic based on position relative to quartiles
    def cat_score(v):
        if v <= Q1:
            return -1  # negative
        elif v >= Q3:
            return 1   # positive
        else:
            return 0   # neutral

    # apply classification to the new column of the df
    df['valence_cat'] = df['valence'].apply(cat_score)

    return df

### Categorization for multiple participants

In [ ]:
def categorize_valence_multiple(df):

    all_words = list(w for w in df['word'].unique())
            
    print(f"Number of unique words: {len(all_words)}")
    valence_map = {} 

    for word in all_words: 
        
        # all the valences for one word
        keyword_valence = df['valence'][df['word'] == word]
        
        # all the valences for the other words
        all_valences = df['valence'][~(df['word'] == word)]
        
        # if the word has less than 3 occurrences (3 scores), assign neutral valence
        count = df['word'].str.count(word).sum()
        if count < 3:
            valence_map[word] = 0

        else:
            # kruskal wallis to see if the two groups are comparable
            stat, p_value = stats.kruskal(keyword_valence, all_valences)
                # if p >= 0.1 --> comparabile
                # if p < 0.1 --> not comparabile
                
            # mean of all the valences of keyword
            mean_keyword_valence = np.mean(keyword_valence)
            # mean of all the valences of the other words
            mean_valences = np.mean(all_valences)
            
            # define the score of the valence based on the p-value found by kruskal wallis
            if p_value < 0.1 and mean_keyword_valence < mean_valences:
                valence_map[word] = -1 # negative
            elif p_value < 0.1 and mean_keyword_valence > mean_valences:
                valence_map[word] = 1 # positive
            elif p_value >= 0.1 or (p_value < 0.1 and mean_keyword_valence == mean_valences):
                valence_map[word] = 0 # neutral

    # add categorization valence column to the df (for each corresponding word)
    df['valence_cat'] = df['word'].map(valence_map)

    # call function to plot valence categorization distribution
    # plot_valence_distribution(df)

    return df

## Forma mentis network building

In [ ]:
def build_multiple_fmnt(df, keywords, folder_path, file_path):
    
    # initialize an empty undirected graph
    G = nx.Graph()

    # create a valence dictionary for all nodes (cue words + responses)
    valence_dict = dict(zip(df['word'], df['valence_cat']))

    # create sets for negative, positive and neutral words
    positive_words = set()
    negative_words = set()
    neutral_words = set()
    
    # add all unique words as nodes with their valence category as a node attribute
    for node, val in valence_dict.items(): 
        G.add_node(node, valence=val)
        if val == 1:
            positive_words.add(node)
        elif val == -1:
            negative_words.add(node)
        elif val == 0:
            neutral_words.add(node)

    custom_valences = [positive_words, negative_words, neutral_words]

    # add undirected edges between cue words and their responses (for each row)
    edges = [(cue, word) for cue, word in zip(df['cue_word'], df['word']) if cue != word]
    G.add_edges_from(edges)

    # create a Formamentis object from the graph
    fmn = emo.nxgraph_to_formamentis(G)

    # create a subgraph for each of the keywords we are intereseted in and calculate tehir features
    for keyword in keywords:
        try: 
            local_clustering_coeff = nx.clustering(G, keyword)
            print(f"Local clustering {keyword}: {local_clustering_coeff}")
            degree = G.degree(keyword)
            print(f"Degree {keyword}: {degree}")

            # plot bfmn
            fig, ax = plt.subplots(figsize=(14,14))
            ax.axis('off')       
            fmnt_keyword = emo.extract_word_from_formamentis(fmn, keyword)
            emo.draw_formamentis(
            fmn=fmnt_keyword,
            layout="edge_bundling",
            # thickness=2,
            custom_valences=custom_valences, 
            ax=ax,
            #alpha_syntactic=1.0,
            )
            fig.savefig(f'{folder_path}/out/analysis/bfmn/{file_path}/{keyword}.png', bbox_inches="tight", pad_inches=0.1)
            plt.close(fig)
        
            # get all the vertices (words) of the subgraph
            vertices = fmnt_keyword.vertices
            # count the number of words in each category
            count_pos = 0
            count_neg = 0
            count_neu = 0
            for word in vertices:
                if word in positive_words:
                    count_pos += 1
                elif word in negative_words:
                    count_neg += 1
                elif word in neutral_words:
                    count_neu += 1
            # create a dictionary to count the number of words in each category and find the mode
            counts = {
                'positive': count_pos,
                'negative': count_neg,
                'neutral': count_neu
            }
            print(f"Counts of '{keyword}' = {counts}")
            print(f"Mode of '{keyword}' = {max(counts, key=counts.get)}")

            ### !!! OPTIONAL: PLOT EMOTIONAL FLOWER !!!
            zeta_scores = emo.zscores(fmnt_keyword)
            emo.draw_plutchik(zeta_scores, reject_range=(-1.96, 1.96))
            plt.savefig(f"{folder_path}/out/analysis/emotional_flowers/{file_path}/{keyword}", bbox_inches="tight", pad_inches=0.1)
            plt.close('all')
            
        except ValueError as e:
            if "min() arg is an empty sequence" in str(e):
                print(f"⚠️ Skipping {keyword}: not enough data to build the network.")
                     
    return G, fmn

In [ ]:
def build_single_fmnt(df):
    
    # initialize an empty undirected graph
    G = nx.Graph()

    # create a valence dictionary for all nodes (cue words + responses)
    valence_dict = dict(zip(df['word'], df['valence_cat']))

    # create lists for negative, positive and neutral words
    positive = []
    negative = []
    neutral = []
    
    # add all unique words as nodes with their valence category as a node attribute
    for node, val in valence_dict.items(): 
        G.add_node(node, valence=val)
        if val == 1:
            positive.append(node)
        elif val == -1:
            negative.append(node)
        elif val == 0:
            neutral.append(node)

    # add undirected edges between cue words and their responses (for each row)
    edges = list(zip(df['cue_word'], df['word']))
    G.add_edges_from(edges)

    # get the valence, degree and closeness centrality of the words of interest
    features_fmnt_dict = {}
    for word in ['ansia', 'matematica']:
        valence = 1 if word in positive else -1 if word in negative else 0 if word in neutral else None
        degree = G.degree(word)
        clo_cen = nx.closeness_centrality(G, word)

        features_fmnt_dict[f'valence_{word}'] = valence
        features_fmnt_dict[f'degree_{word}'] = degree
        features_fmnt_dict[f'clo_cen_{word}'] = clo_cen

    # wrap in list to create a single-row DataFrame
    features_fmnt = pd.DataFrame([features_fmnt_dict])
    # add participant_id to the DataFrame (first col)
    features_fmnt.insert(loc=0, column='participant_id', value=df['participant_id'].iloc[0])  
        
    return G, features_fmnt

## Pipelines

In [ ]:
# function to use on the data analysis of multiple partcipants
def pipeline_multiple_participants(df, keywords, folder_path, file_path):
    clean_df = text_cleaning(df)
    long_df = build_association_df(clean_df)
    final_df = categorize_valence_multiple(long_df)
    G, fmn = build_multiple_fmnt(final_df, keywords, folder_path, file_path)
    return final_df, G, fmn

In [ ]:
def pipeline_multiple_df(df):
    clean_df = text_cleaning(df)
    long_df = build_association_df(clean_df)
    final_df = categorize_valence_multiple(long_df)
    return final_df

In [ ]:
# function to use on the data analysis of a single partcipant
def pipeline_single_participant(df):
    clean_df = text_cleaning(df)
    long_df = build_association_df(clean_df)
    final_df = categorize_valence_single(long_df)
    G, features_fmnt  = build_single_fmnt(final_df)
    return final_df, G, features_fmnt

In [ ]:
def run_pipeline_all_participants(df):
    results = []
    # usage of the single participant pipeline on all the participants of a df (gpt_df) 
    groups_total = df.groupby('participant_id')
    
    for participant_id, participant_df in groups_total:
        final_df, G, features_fmnt_df = pipeline_single_participant(participant_df)
        # append the df with the results of the features of interest (degree, valence, closeness centrality) to the results list
        results.append(features_fmnt_df)

    # transform the list of results into a single DataFrame
    return pd.concat(results, ignore_index=True)

# Pipeline application on desired dataset

## Dataset reading

In [ ]:
path_math_anxiety_llms = "/Users/emma/projects/cognoscolab/math_anxiety_llms"
path_rassereno = '/Users/emma/projects/cognoscolab/rassereno'

In [ ]:
splitted_df_path = f'{path_math_anxiety_llms}/out/splitted_dfs/'
# gets high and low anxiety gpt participants
high_anxiety_gpt_4o = pd.read_csv(f'{splitted_df_path}/high_anxiety_gpt_4o.csv')
low_anxiety_gpt_4o = pd.read_csv(f'{splitted_df_path}/low_anxiety_gpt_4o.csv')
total_gpt_4o = pd.read_csv(f'{path_math_anxiety_llms}/out/formatting/gpt_4o/associations_valence_gpt_4o.csv')

high_anxiety_gpt_3_5 = pd.read_csv(f'{splitted_df_path}/high_anxiety_gpt_3_5.csv')
low_anxiety_gpt_3_5 = pd.read_csv(f'{splitted_df_path}/low_anxiety_gpt_3_5.csv')
total_gpt_3_5 = pd.read_csv(f'{path_math_anxiety_llms}/out/formatting/gpt_3_5/associations_valence_gpt_3_5.csv')

# gets high and low anxiety 2024 humans participants
high_anxiety_2024_humans = pd.read_csv(f'{splitted_df_path}/high_anxiety_2024_humans.csv')
low_anxiety_2024_humans = pd.read_csv(f'{splitted_df_path}/low_anxiety_2024_humans.csv')
total_2024_humans = pd.read_csv(f'{path_math_anxiety_llms}/out/formatting/humans_2024/associations_valence_2024_humans.csv')

total_2023_humans = pd.read_csv(f'{path_math_anxiety_llms}/out/formatting/humans_2023/associations_valence_2023_humans.csv')

In [ ]:
high_anxiety_all_university = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/high_anxiety_all_university_students.csv')
low_anxiety_all_university = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/low_anxiety_all_university_students.csv')
liceo_2025 = pd.read_csv(f'{path_rassereno}/out/formatting/liceo_2025/associations_valence_liceo_2025.csv')
students_2021 = pd.read_csv(f'{path_rassereno}/out/formatting/students_2021/associations_valence_students_2021.csv')
experts_2021 = pd.read_csv(f'{path_rassereno}/out/formatting/experts_2021/associations_valence_experts_2021.csv')

## Rassereno: multiple participants bfmn

In [ ]:
# keywords that we want to be plotted
keywords_math_llm = ['statistica', 'scienza', 'matematica', 'terapeuta', 'ansia']
keywords_rassereno = ['ricerca', 'scuola', 'informatica', 'studente', 'voto', 'insegnante', 'professore', 'matematica', 'scienza', 'fisica', 'statistica', 'università', 'esame', 'scienziato']
keywords_rassereno_eng = ['research', 'school', 'computer', 'student', 'grade', 'teacher', 'professor', 'math', 'science', 'physics', 'statistics', 'university', 'exam', 'scientist']

### Dataframe preparation

In [ ]:
df_experts_2021 = categorize_valence_multiple(experts_2021)

In [ ]:
df_high_school_2021 = categorize_valence_multiple(students_2021)

In [ ]:
df_high_school_2025 = pipeline_multiple_df(liceo_2025)

In [ ]:
df_high_anxiety_all_university = pipeline_multiple_df(high_anxiety_all_university)

In [ ]:
df_low_anxiety_all_university = pipeline_multiple_df(low_anxiety_all_university)

In [ ]:
df_low_anxiety_gpt_4o = pipeline_multiple_df(low_anxiety_gpt_4o)

In [ ]:
df_high_anxiety_gpt_4o = pipeline_multiple_df(high_anxiety_gpt_4o)

### BFMN building

In [ ]:
G, fmn = build_multiple_fmnt(df_experts_2021, keywords_rassereno_eng, path_rassereno, 'experts_2021')

In [ ]:
G, fmn = build_multiple_fmnt(df_high_school_2021, keywords_rassereno, path_rassereno, 'liceo_2021')

In [ ]:
G, fmn = build_multiple_fmnt(df_high_school_2025, keywords_rassereno, path_rassereno, 'liceo_2025')

In [ ]:
G, fmn = build_multiple_fmnt(df_high_anxiety_all_university, keywords_rassereno, path_rassereno,'university_2025/high_anxiety')

In [ ]:
G, fmn = build_multiple_fmnt(df_low_anxiety_all_university, keywords_rassereno, path_rassereno,'university_2025/low_anxiety')

In [ ]:
G, fmn = build_multiple_fmnt(df_low_anxiety_gpt_4o, keywords_rassereno, path_rassereno,'gpt-4o/low_anxiety')

In [ ]:
G, fmn = build_multiple_fmnt(df_high_anxiety_gpt_4o, keywords_rassereno, path_rassereno,'gpt-4o/high_anxiety')

## Math anxiety in LLMs: Single participant pipeline

In [ ]:
features_gpt_3_5 = run_pipeline_all_participants(total_gpt_3_5)

In [ ]:
features_gpt_4o = run_pipeline_all_participants(total_gpt_4o)

In [ ]:
features_humans_2024 = run_pipeline_all_participants(total_2024_humans)

In [ ]:
features_humans_2023 = run_pipeline_all_participants(total_2023_humans)

## Math anxiety in LLMs: multiple BFMN

### GPT-3.5

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(high_anxiety_gpt_3_5, keywords_math_llm, 'gpt_3_5/high_anxiety')

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(low_anxiety_gpt_3_5, keywords_math_llm, 'gpt_3_5/low_anxiety')

### GPt-4o

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(total_gpt_4o, keywords_math_llm, 'gpt_4o')

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(high_anxiety_gpt_4o, keywords_math_llm, 'gpt_4o/high_anxiety')

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(low_anxiety_gpt_4o, keywords_math_llm, 'gpt_4o/low_anxiety')

### 2024 humans

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(high_anxiety_2024_humans, keywords_math_llm, 'humans_2024/high_anxiety')

In [ ]:
final_df, G, fmn = pipeline_multiple_participants(low_anxiety_2024_humans, keywords_math_llm, 'humans_2024/low_anxiety')

# Linear regressions and descriptive stats

## Network features descriptive stats

In [ ]:
mas_it_factors_path = f'{path_math_anxiety_llms}/out/mas_it_factors'
features_bfmn_path = f'{path_math_anxiety_llms}/out/analysis/features_bfmn'

In [ ]:
def network_features(sample, features):
    mas_it_factors = pd.read_csv(f'{mas_it_factors_path}/mas_it_factors_{sample}.csv')

    features_total= mas_it_factors.merge(features, on='participant_id', how='left')
    features_total = features_total.drop(columns=['participant_id'])
    features_total.to_csv(f'{features_bfmn_path}/features_total_{sample}.csv', index=False)

    results = features_total.describe().T.drop(columns=['25%','50%', '75%', 'min', 'max'])
    results = results.drop(index=['mas_total_ma', 'mas_social', 'mas_eval', 'mas_pass_obs', 'valence_ansia', 'valence_matematica'])
    row_names = {
            'degree_ansia': '"Anxiety" Degree',
            'clo_cen_ansia': '"Anxiety" Closeness Centrality',
            'degree_matematica': '"Math" Degree',
            'clo_cen_matematica': '"Math" Closeness Centrality'
        }
    results.index = results.index.map(row_names)
    latex_table = results.to_latex(escape=False, float_format="%.2f")

    print(latex_table)
    return features_total

In [ ]:
features_total_gpt_3_5 = network_features('gpt_3_5', features_gpt_3_5)

In [ ]:
features_total_gpt_4o = network_features('gpt_4o', features_gpt_4o)

In [ ]:
features_total_2024_humans = network_features('2024_humans', features_humans_2024)

In [ ]:
features_total_2023_humans = network_features('2023_humans', features_humans_2023)

## Correlograms

In [ ]:
def plot_correlogram(df, file_name):
    df_copied = df.copy()
    df_copied = df_copied[['mas_total_ma', 'mas_eval', 'mas_social', 'mas_pass_obs',
                            'valence_ansia', 'degree_ansia', 'clo_cen_ansia',
                            'valence_matematica', 'degree_matematica', 'clo_cen_matematica']]
    # change col names for visualization
    df_copied.columns = [
    "Total MA",
    "Evaluation MA",
    "Social MA",
    "Observation MA",
    "Valence \"anxiety\"",
    "Degree \"anxiety\"",
    "CC \"anxiety\"",
    "Valence \"math\"",
    "Degree \"math\"",
    "CC \"math\""
    ]

    # compute pearson correlation matrix
    corr = df_copied.corr(method='pearson')

    # inizialize df for p-values
    annot = corr.copy()

    # compute p-values for each pair of variables, with their signficance stars
    for i in corr.columns:
        for j in corr.columns:
            if i != j:
                _, p = stats.pearsonr(df_copied[i], df_copied[j])

                if p < 0.001:
                    p = '***'
                elif p < 0.01:
                    p = '**'
                elif p < 0.05:
                    p = '*'
                else:
                    p = ''
                annot.loc[i, j] = f"{corr.loc[i, j]:.2f}{p}"
            else:
                annot.loc[i, j] = f"{corr.loc[i, j]:.2f}"

    # plot the correlogram
    plt.figure(figsize=(12, 10))
    # keep only the upper triangle
    matrix = np.triu(np.ones_like(corr, dtype=bool))
    # plot heatmap with corr values, p-values with their stars, and the bar for colors
    sns.heatmap(corr, annot=annot, annot_kws={"size":12},fmt='', cmap="crest", 
                mask = matrix,
                cbar_kws={"orientation": "horizontal", 'shrink': 0.6, 'location' : 'top', "pad": 0}
                )
    # rotate the x and y ticks for better visualization
    plt.xticks(rotation=45, ha='right')   
    plt.yticks(rotation=45, ha='right')    
    plt.tight_layout()
    # save the plot
    plt.savefig(f'{path_math_anxiety_llms}/out/analysis/correlograms/{file_name}.png', bbox_inches="tight")
    plt.close()

In [ ]:
# plot the correlogram for each participants sample
plot_correlogram(features_total_2023_humans, 'humans_2023')
plot_correlogram(features_total_2024_humans, 'humans_2024')
plot_correlogram(features_total_gpt_3_5, 'gpt_3_5')
plot_correlogram(features_total_gpt_4o, 'gpt_4o')

## Standardize values

In [ ]:
def stand_vars_lin(data):

    scaler = MinMaxScaler()

    lin_reg_vars = data[['mas_total_ma', 'mas_social', 'mas_eval', 'mas_pass_obs', 'valence_ansia', 'degree_ansia', 'valence_matematica', 'degree_matematica']].copy()

    stand_data = pd.DataFrame()
    for col in lin_reg_vars.columns:
        scaled_col = scaler.fit_transform(lin_reg_vars[[col]])
        stand_data[col] = scaled_col.flatten()

    return stand_data

In [ ]:
stand_data_gpt_3_5 = stand_vars_lin(features_total_gpt_3_5)
stand_data_gpt_4o = stand_vars_lin(features_total_gpt_4o)
stand_data_2024_humans = stand_vars_lin(features_total_2024_humans)
stand_data_2023_humans = stand_vars_lin(features_total_2023_humans)

## Linear regression OLS models

### Functions

In [ ]:
def lin_reg_ols_predictors(stand_data, factor, predictors):
    # depentent variable
    Y = stand_data[factor]
    # parameters
    X = stand_data[predictors]
    # intercept
    X = sm.add_constant(X) 

    model = sm.OLS(Y, X).fit()

    results = model.summary2().tables[1]

    # function to format p-values with asterisks
    def format_p(p):
        formatted_p = f"{p:.3f}".lstrip("0") # keeps 3 decimals and removes leading zeros
        if p < 0.001:
            return '< .001***'
        if p < 0.01:
            return f"{formatted_p}**"
        elif p < 0.05:
            return f"{formatted_p}*"
        else:
            return formatted_p

    # take only the columns of interest
    results = results[['Coef.', 'Std.Err.', 't', 'P>|t|']]
    # keep the first 3 decimals of the coefficients, standard errors and t-values
    col_to_format = ['Coef.', 'Std.Err.', 't']
    # for all the results keep only 3 decimals and remove leading zeros
    results[col_to_format] = results[col_to_format].applymap(lambda x: f"{x:.3f}".lstrip("0"))
    # apply the p-value formatting function to the 'P>|t|' column
    results['P>|t|'] = results['P>|t|'].apply(format_p)

    # rename the indices of the table based on the predictors used
    if len(predictors) == 4:
        results.index = ['Intercept', '"Anxiety" Degree', '"Math" Degree', '"Anxiety" Valence', '"Math" Valence']
    elif len(predictors) == 3:
        results.index = ['Intercept', '"Anxiety" Degree', '"Anxiety" Valence', '"Math" Valence']
        
    # build latex table to be inserted in overleaf
    latex_table = results.to_latex(escape=False)  # keep asterisks

    # print the R-squared value and latex table
    print(f"R-squared {factor} {len(predictors)} predictors: {model.rsquared:.3f}\nAIC {factor} {len(predictors)} predictors: {model.aic}\n")
 
    # if we wanto to see the latex table we can print it
    return model.aic, latex_table

In [ ]:
factors = ['mas_total_ma', 'mas_eval', 'mas_social', 'mas_pass_obs']

In [ ]:
def lin_regression_results(stand_data, model_name):
    for factor in factors:
        print(f'## Results for {model_name} {factor} ##\n')
        aic_4_factors, table_4_factors = lin_reg_ols_predictors(stand_data, factor, ['degree_ansia', 'degree_matematica', 'valence_ansia','valence_matematica'])
        aic_3_factors, table_3_factors = lin_reg_ols_predictors(stand_data, factor, ['degree_ansia', 'valence_ansia', 'valence_matematica']) # perché negli umani il degree di math non correla con i fattori della math anxiety

        if aic_4_factors < aic_3_factors:
            print("--> 4 predictors model is better\n")
            print(table_4_factors)
        else:
            print("--> 3 predictors model is better\n")
            print(table_3_factors)

### gpt-4o

In [ ]:
lin_regression_results(stand_data_gpt_4o, 'GPT-4o')

### gpt-3.5

In [ ]:
lin_regression_results(stand_data_gpt_3_5, 'GPT-3.5')

### 2024 humans

In [ ]:
lin_regression_results(stand_data_2024_humans, 'Humans 2024')

### 2023 humans

In [ ]:
lin_regression_results(stand_data_2023_humans, 'Humans 2023')